В цьому домашньому завданні ми проведемо додаткові експерименти для рішення задачі бінарної класифікації і створимо ваш новий submission на змагання на Kaggle.

-----------


**Завдання 0**. Завантажте дані `train.csv`, `test.csv`, `sample_submission.csv` зі змагання на Kaggle - шукайте посилання в уроці [Запрошення до участі у Kaggle-змаганні.](https://data-loves.kwiga.com/courses/machine-learning-dlia-liudei/domashnie-zavdannia-zmagannia-z-kaggle)  Для завантаження потрібно долучитись до змагання (натиснути кнопку "Join").


**Завдання 1**. **Збираємо весь код з попереднього ДЗ в одному місці.** В лекційному ноутбуці `Логістична регресія з ScikitLearn. Повна ML задача.ipynb` ми познайомились з поняттям пайплайнів, а також я показала, як компактно виглядає рішення МЛ задачі, якщо ми зберемо весь код разом.

Оскільки ми далі будемо робити експерименти, які включають ті самі етапи попередньої обробки, але інше моделювання - буде зручно мати весь код компактно і під рукою. Тому зараз ми займемось збором коду до купи :) Після цього завдання для подальших експериментів ви можете перенести частини розвʼязку взагалі в окремий `.py` файл, аби було зручно імпортувати функції.

Зі свого рішення в попередньому домашньому завданні (`Логістична регресія з scikit learn.ipynb`) зберіть усі кроки розвʼязку задачі разом з використанням `sklearn.Pipeline` за прикладом з лекції.

Ваш код нижче має містити
1. Читання даних з файлу (поза пайплайном).
2. Розбиття на тренувальний і валідаційний набори, де валідаційний містить 20% даних (поза пайплайном).
3. Виділення категоріальних і числових колонок (поза пайплайном).
4. Підготовку категоріальних і числових колонок (частина пайплайну). В прикладі в лекції ми оформлювали обробку числових і категоріальних колонок в окремі трансформери `numeric_transformer`, `categorical_cols`. Рекоемндую зробити саме так, так потім зручніше вносити зміни :)
5. Тренування лог регресії (частина пайплайну).
6. Запуск пайплайну на тренування на трен. даних (поза пайплайном).
7. Запуск пайплайну на передбачення на трен і вал. даних і вимір метрик якості ROC-AUC + вивдення Confusion Matrix (поза пайплайном).
8. Збереження моделі в формат joblib (поза пайплайном).

Ви це все вже зробили в попереднтьому ДЗ! Тож, тут просто заадча все зібрати разом.

Нижче я додала підказки, що покроково ви маєте зробити. Якщо ви почуваєтесь впевнено, можете видалити ці підказки і реалізувати все самостійно, або ж - просто заповнити пропуски.

Завдання оцінюється в 10 балів. Головний результат - аби код в фіналі був робочий. Бо за не робочий нам гроші ніхто не заплатить :)

In [24]:
!pip install opendatasets --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import opendatasets as od
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.metrics import roc_curve, auc
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib

# Download the dataset
od.download('https://www.kaggle.com/competitions/bank-customer-churn-prediction-dlu-course-c-4/data')
raw_df = pd.read_csv('./bank-customer-churn-prediction-dlu-course-c-4/train.csv')

# Create training, validation sets
train, valid = train_test_split(
    raw_df, test_size=0.2, random_state=12, stratify=raw_df['Exited'])

# Create inputs and targets
input_cols = train.columns.drop('Exited')
target_col = 'Exited'
train_inputs, train_targets = train[input_cols], train[target_col]
val_inputs, val_targets = valid[input_cols], valid[target_col]

# Identify columns
drop_cols = ['id', 'CustomerId', 'Surname']
categorical_cols = ['Geography', 'Gender']
scale_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']
binary_cols = ['HasCrCard', 'IsActiveMember']

model_input_cols = [c for c in input_cols if c not in drop_cols]

# Create preprocessing pipelines for both numeric and categorical data
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

# Combine transformers into a preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, scale_cols),
        ('cat', categorical_transformer, categorical_cols),
        ('bin', 'passthrough', binary_cols),
    ],
    remainder='drop'
)

# Create a pipeline that includes preprocessing and the model
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(solver='liblinear', random_state=12))
])

model_pipeline.fit(train_inputs[model_input_cols], train_targets)

# Helper function to predict, compute accuracy & plot confusion matrix
def evaluate(model_pipeline, inputs, targets, name=''):
  proba = model_pipeline.predict_proba(inputs)[:, 1]
  preds = (proba >= 0.5).astype(int)

  print('----------------------------------')
  print(f'{name}')
  print(f'F1 score : {f1_score(targets, preds):.3f}')
  print(f'ROC-AUC  : {roc_auc_score(targets, proba):.3f}')
  print('Confusion matrix:')
  print(confusion_matrix(targets, preds, normalize='true').round(3))

# Evaluate on train/validation
train_preds = evaluate(model_pipeline, train_inputs[model_input_cols], train_targets, 'Train')
val_preds = evaluate(model_pipeline, val_inputs[model_input_cols], val_targets, 'Validation')

# Save model
artifact = {
    'pipeline': model_pipeline,
    'model_input_cols': model_input_cols
}
joblib.dump(artifact, 'log_reg_pipeline.joblib')

Skipping, found downloaded files in "./bank-customer-churn-prediction-dlu-course-c-4" (use force=True to force download)
----------------------------------
Train
F1 score : 0.634
ROC-AUC  : 0.882
Confusion matrix:
[[0.957 0.043]
 [0.458 0.542]]
----------------------------------
Validation
F1 score : 0.652
ROC-AUC  : 0.881
Confusion matrix:
[[0.954 0.046]
 [0.428 0.572]]


['log_reg_pipeline.joblib']

**Завдання 2**. Такс, у нас з вами є вже готовий пайплайн. Давайте проведемо нові експерименти.

  Додайте в попередню обробку числових колонок генерацію polinomal features до степені 2 включно. Для цього створіть новий препроцесор і створіть новий пайплайн.

  Запустіть пайплайн на тренування і виведіть метрики для тренувального і валідаційного набору. Напишіть, як вам модель? Чи спостерігається в цій моделі overfit чи underfit? Чи ця модель добре генералізує?

In [2]:
from sklearn.preprocessing import PolynomialFeatures

numeric_transformer_poly_2 = Pipeline(steps=[
    ('poly_2', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler())
])

preprocessor_poly_2 = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_poly_2, scale_cols),
        ('cat', categorical_transformer, categorical_cols),
        ('bin', 'passthrough', binary_cols),
    ],
    remainder='drop'
)

model_pipeline_poly_2 = Pipeline(steps=[
    ('preprocessor', preprocessor_poly_2),
    ('classifier', LogisticRegression(solver='liblinear', max_iter=2000, random_state=12))
])

model_pipeline_poly_2.fit(train_inputs[model_input_cols], train_targets)

evaluate(model_pipeline_poly_2, train_inputs[model_input_cols], train_targets, 'Train (poly_2)')
evaluate(model_pipeline_poly_2, val_inputs[model_input_cols], val_targets, 'Validation (poly_2)')


Train (poly_2)
F1 score : 0.728
ROC-AUC  : 0.928
Confusion matrix:
[[0.955 0.045]
 [0.327 0.673]]
Validation (poly_2)
F1 score : 0.739
ROC-AUC  : 0.926
Confusion matrix:
[[0.956 0.044]
 [0.315 0.685]]


Додавання поліноміальних ознак  суттєво покращило якість моделі ROC-AUC зріс до 0.926, F1 до 0.739 (на validation). Різниця між метриками на train і validation мінімальна, тому ознак overfitting не спостерігається. Модель добре узагальнює

**Завдання 3**. Тепер давайте створимо ще новий пайплайн, тільки тепер поліноміальні ознаки згенеруємо до степені 4. Зробіть висновок про якість моделі. Якщо вам подобається резульат якоїсь з моделей в цьому ДЗ - рекомендую зробити submission в змаганні.

In [3]:
numeric_transformer_poly_4 = Pipeline(steps=[
    ('poly_4', PolynomialFeatures(degree=4, include_bias=False)),
    ('scaler', StandardScaler())
])

preprocessor_poly_4 = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_poly_4, scale_cols),
        ('cat', categorical_transformer, categorical_cols),
        ('bin', 'passthrough', binary_cols),
    ],
    remainder='drop'
)

model_pipeline_poly_4 = Pipeline(steps=[
    ('preprocessor', preprocessor_poly_4),
    ('classifier', LogisticRegression(solver='liblinear', max_iter=5000, random_state=12))
])

model_pipeline_poly_4.fit(train_inputs[model_input_cols], train_targets)

evaluate(model_pipeline_poly_4, train_inputs[model_input_cols], train_targets, 'Train (poly_4)')
evaluate(model_pipeline_poly_4, val_inputs[model_input_cols], val_targets, 'Validation (poly_4)')

Train (poly_4)
F1 score : 0.737
ROC-AUC  : 0.934
Confusion matrix:
[[0.957 0.043]
 [0.319 0.681]]
Validation (poly_4)
F1 score : 0.739
ROC-AUC  : 0.932
Confusion matrix:
[[0.957 0.043]
 [0.315 0.685]]


Додавання поліноміальних ознак до степені 4 дещо покращило якість моделі ROC-AUC = 0.932 (на validation). Метрики на train і validation практично однакові, тому переобучення не спостерігається, модель добре узагальнює. Порівняно з degree=2, приріст ROC-AUC невеликий

**Завдання 4. Перенавчання і регуляризація**.

  Скачайте набір даних `regression_data.csv`. Звичайте набір даних з `regression_data.csv`, розбийте на train і test (в тест 20%) і натренуйте модель лінійної регресії з масштабуванням числових ознак і поліноміальними ознаками до степені **5 включно**.

  Виміряйте якість прогностичної моделі і зробіть висновок, чи модель хороша, чи вона добре генералізує?


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
data = pd.read_csv("drive/MyDrive/Colab Notebooks/regression_data.csv")
data.head()

,feature_1,feature_2,feature_3,feature_4,feature_5,target
0,-0.190339,-1.382800,-0.875618,0.538910,-1.037246,28.938854
1,-0.321386,-0.563725,0.412931,-0.147057,-0.825497,-7.664581
2,2.122156,-1.519370,1.032465,-1.260884,0.917862,-63.845482
3,-1.380101,-0.055548,-1.703382,0.074095,1.628616,4.076259
4,-0.072829,-1.514847,-0.846794,0.714000,0.473238,34.879013


In [6]:
train, test = train_test_split(data, test_size=0.2, random_state=12)

input_cols = list(train.columns)[1:-1]
target_col = 'target'

train_inputs, train_targets = train[input_cols], train[target_col]
val_inputs, val_targets = test[input_cols], test[target_col]

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

lr = LinearRegression().fit(train_inputs, train_targets)

prediction_train = lr.predict(train_inputs)
prediction_val = lr.predict(val_inputs)

loss_train = np.sqrt(mean_squared_error(train_targets, prediction_train))
loss_test = np.sqrt(mean_squared_error(val_targets, prediction_val))

print(loss_train, loss_test)

1.0126018746590522 1.1456776591565003


In [8]:
import operator
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

#polynomial_features
polynomial_features = PolynomialFeatures(degree=5)
train_inputs_poly = polynomial_features.fit_transform(train_inputs)
val_inputs_poly = polynomial_features.transform(val_inputs)

#scaling
scaler = StandardScaler().fit(train_inputs_poly)
train_inputs_poly = scaler.transform(train_inputs_poly)
val_inputs_poly = scaler.transform(val_inputs_poly)

#training
lr_poly = LinearRegression().fit(train_inputs_poly, train_targets)

prediction_train_poly = lr_poly.predict(train_inputs_poly)
prediction_val_poly = lr_poly.predict(val_inputs_poly)

loss_train_poly = np.sqrt(mean_squared_error(train_targets, prediction_train_poly))
loss_test_poly = np.sqrt(mean_squared_error(val_targets, prediction_val_poly))

print(loss_train_poly, loss_test_poly)

1.340674600137576e-13 71.74589809192847


Error на validation сильно збільшився, що свідчить про сильний overfitting.
Модель з поліноміальними ознаками степеня 5 не узагальнює добре і запам’ятовує тренувальні дані

**Завдання 5**. Натренуйте моделі Lasso(), Ridge(), ElasaticNet() на цих даних (з поліном ознаками до степені 20 включно), порівняйте якість з тою, яка була отримана з лінійною регресією. Яка модель найкраще генералізує і чому на ваш погляд (можливо треба буде для відповіді зробити додатковий аналіз ознак)?

In [9]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
models = [
    Lasso(),
    Ridge(),
    Ridge(alpha=2),
    ElasticNet(),
    ElasticNet(alpha=0.5)
]

In [10]:
def evaluate_model(model, X_train, y_train, X_val, y_val):
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)

    train_metrics = np.sqrt(mean_squared_error(y_train, y_train_pred)).round(6)
    val_metrics = np.sqrt(mean_squared_error(y_val, y_val_pred)).round(6)

    return dict(train=train_metrics, val=val_metrics)

In [11]:
poly_features_20 = PolynomialFeatures(degree=20)
train_inputs_poly_20= poly_features_20.fit_transform(train_inputs)
val_inputs_poly_20 = poly_features_20.transform(val_inputs)

In [12]:
scaler = StandardScaler()
train_inputs_poly_20 = scaler.fit_transform(train_inputs_poly_20)
val_inputs_poly_20 = scaler.transform(val_inputs_poly_20)

In [13]:
train_inputs_poly_20.shape

(103, 10626)

In [14]:
import warnings
warnings.filterwarnings('ignore')

In [15]:
for model in models:
    model.fit(train_inputs_poly_20,train_targets)
    eval_results  = evaluate_model(model, train_inputs_poly_20, train_targets, val_inputs_poly_20, val_targets)
    print(f'{str(model)}: {eval_results}\n')

Lasso(): {'train': np.float64(1.43638), 'val': np.float64(1.857817)}

Ridge(): {'train': np.float64(1.195105), 'val': np.float64(1828.207292)}

Ridge(alpha=2): {'train': np.float64(1.829228), 'val': np.float64(1941.617082)}

ElasticNet(): {'train': np.float64(12.543019), 'val': np.float64(60.510846)}

ElasticNet(alpha=0.5): {'train': np.float64(8.696062), 'val': np.float64(224.876814)}



Якість моделей на датасеті з поліном ознаками до степені 20 включно, дуже відрізняється, наприклад Lasso добре генералізує, ElasticNet() генералізує краще ніж LinearRegression на поліноміальних ознаках до 5 ступеня включно, але всі інші моделі - значно гірше. Що в принципі зрозуміло - у нас в датасеті 103 строчки і 10626 фічі, оверфіттінг очікуваний. Lasso генералізує найкраще, бо зануляє частину коефіцієнтів

In [16]:
np.sort(models[0].coef_.round(5))

array([ 0.     , -0.     , -0.     , ..., -0.     ,  0.     , 42.55208])

In [17]:
np.sort(models[1].coef_.round(5))

array([-1.55564, -1.53713, -1.24774, ...,  6.32617,  6.49994, 35.06811])

In [18]:
np.sort(models[2].coef_.round(5))

array([-1.61327, -1.5504 , -1.20953, ...,  6.2307 ,  7.33357, 32.86142])